In [1]:
%load_ext autoreload
%autoreload 2
# %matplotlib inline

import os
while 'notebooks' in os.getcwd():
    os.chdir("../")

import torch
from torch import nn, einsum

import quantus
import gc
import torch.nn.functional as F
import pandas as pd

from lib.helpers import plot_example_grid
from lib.attributions import GradientAscentDiff, PullbackAscentDiff, \
    quantus_pullback_ascent_diff_explain_func
from lib.setup import setup_notebook
from lib.defaults import get_default_kwargs
from lib.surrogates import LayerNorm2d, PVTAttention, soften_module_inplace_
from lib.evaluator import QuantusEvaluator, default_explainers, default_metrics

c:\Users\Maciej\Documents\GitHub\SemanticPullbacks\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
short_metrics_map = {
    "infidelity": "Infidelity",
    "faithfulness_correlation": "Faith.Corr",
    "faithfulness_estimate": "Faith.Est",
    "monotonicity_correlation": "Mono.Corr",
    "max_sensitivity": "Max.Sens",
    "random_logit": "Rand.Logit",
}
explainers = ["SoftPullback", "PullbackAscent", "SmoothPullback", "FusionPullback", "Gradient", "GradientAscent", "SmoothGrad", "FusionGrad", "GradientShap", "IntegratedGradients", "DeepLift", "GuidedGradCam"]
# explainers = ["SoftPullback", "PullbackAscent", "PullbackAscentNoAlpha", "PullbackAscent3", "SmoothPullback", "FusionPullback", "Gradient", "GradientAscent", "SmoothGrad", "FusionGrad", "GradientShap", "IntegratedGradients", "DeepLift", "GuidedGradCam"]

In [9]:
def merge_with_priority(df_big, df_small, key_col="Faith.Corr"):
    # we merge results as we re-genenerated the faithfulness_correlation metric for different set of patches
    # the previous selection had too large patches, which inflated the correlation score
    result = df_big.loc[df_small.index].copy()
    result[key_col] = df_small[key_col]
    return result

def process_df(df_path, precision=3):
    selected_columns = list(short_metrics_map.keys())
    
    df = QuantusEvaluator.load_results(df_path)
    # df = df[selected_columns].rename(index=short_metrics_map, columns=short_metrics_map)
    df = df[
        [col for col in selected_columns if col in df.columns]
    ].rename(index=short_metrics_map, columns=short_metrics_map)
    # df = df.loc[explainers]
    df = df.loc[[idx for idx in explainers if idx in df.index]]
    
    print("num_samples:", len(df.iloc[0].iloc[0]))
    
    df = QuantusEvaluator.summarize_results(df, precision=precision)
    return df


In [10]:
vgg_df = process_df("results/quantus_vgg_fc_nips_20_50_vgg11_bn")
vgg_df.to_csv('results/vgg_df_20_50.csv', index=True)
vgg_df

num_samples: 1000


,Infidelity,Faith.Corr,Faith.Est,Mono.Corr,Max.Sens,Rand.Logit
SoftPullback,16.303±31.68,0.606±0.3,0.53±0.4,0.375±0.39,0.212±0.08,-0.064±0.33
PullbackAscent,5.616±13.18,0.57±0.29,0.376±0.48,0.312±0.37,0.39±0.11,0.136±0.13
SmoothPullback,14.03±30.57,0.545±0.32,0.428±0.46,0.309±0.4,0.398±0.11,-0.048±0.32
FusionPullback,16.056±34.46,0.513±0.33,0.406±0.47,0.315±0.39,0.513±0.11,-0.035±0.29
Gradient,100.09±107.94,0.457±0.35,0.466±0.41,0.316±0.37,0.898±0.14,-0.053±0.29
GradientAscent,18.523±35.24,0.508±0.3,0.264±0.46,0.234±0.32,1.217±0.06,6.65e-04±0.07
SmoothGrad,52.044±70.8,0.513±0.34,0.508±0.41,0.385±0.36,0.781±0.11,-0.025±0.19
FusionGrad,38.651±60.23,0.506±0.34,0.521±0.39,0.363±0.34,0.955±0.13,-0.013±0.16
GradientShap,76.05±94.76,0.43±0.36,0.632±0.33,0.381±0.41,1.086±0.22,-0.042±0.26
IntegratedGradients,75.796±92.29,0.434±0.36,0.634±0.33,0.379±0.41,0.752±0.15,-0.046±0.27


In [11]:
pvt_df_fc = process_df("results/quantus_pvt_fc_nips_20_50_pvt_v2_b1")
pvt_df = process_df("results/quantus_pvt_it_nips_20_50_pvt_v2_b1")
pvt_merged_df = merge_with_priority(pvt_df, pvt_df_fc)
pvt_merged_df.to_csv('results/pvt_df_20_50.csv', index=True)
pvt_merged_df

num_samples: 1000
num_samples: 1000


,Infidelity,Faith.Corr,Faith.Est,Mono.Corr,Max.Sens,Rand.Logit
SoftPullback,6.264±6.4,0.119±0.34,0.16±0.41,0.164±0.43,1.066±0.18,-0.006±0.35
PullbackAscent,1.634±1.03,0.122±0.36,0.201±0.41,0.219±0.33,0.855±0.12,0.121±0.11
SmoothPullback,4.974±5.46,0.102±0.39,0.146±0.49,0.231±0.38,0.519±0.09,0.008±0.23
FusionPullback,5.156±5.72,0.093±0.39,0.114±0.49,0.2±0.38,0.753±0.11,0.007±0.19
Gradient,8.914±7.89,0.105±0.33,0.117±0.41,0.185±0.43,1.034±0.16,-0.03±0.3
GradientAscent,4.506±4.07,0.118±0.33,0.06±0.42,0.164±0.35,1.242±0.07,0.006±0.06
SmoothGrad,8.798±7.08,0.096±0.38,0.171±0.48,0.28±0.38,0.569±0.1,-0.012±0.13
FusionGrad,6.673±5.6,0.101±0.39,0.144±0.48,0.262±0.38,0.979±0.16,5.73e-04±0.11
GradientShap,12.433±8.45,0.067±0.32,0.177±0.39,0.169±0.45,2.471±1.56,0.003±0.34
IntegratedGradients,12.578±8.39,0.068±0.32,0.157±0.4,0.17±0.45,1.374±0.37,0.008±0.33


In [12]:
resnet_df_fc = process_df("results/quantus_resnet_fc_nips_20_50_resnet50")
resnet_df = process_df("results/quantus_resnet_it_nips_20_50_resnet50")
resnet_merged_df = merge_with_priority(resnet_df, resnet_df_fc)
resnet_merged_df.to_csv('results/resnet_df_20_50.csv', index=True)
resnet_merged_df

num_samples: 1000
num_samples: 1000


,Infidelity,Faith.Corr,Faith.Est,Mono.Corr,Max.Sens,Rand.Logit
SoftPullback,5.989±5.67,0.389±0.37,0.437±0.41,0.42±0.39,0.119±0.04,-0.062±0.42
PullbackAscent,5.384±4.83,0.382±0.37,0.394±0.43,0.421±0.4,0.244±0.09,0.212±0.19
SmoothPullback,8.122±10.13,0.326±0.39,0.321±0.46,0.352±0.4,0.22±0.05,-0.065±0.4
FusionPullback,10.131±14.55,0.306±0.4,0.306±0.47,0.35±0.4,0.362±0.08,-0.051±0.39
Gradient,83.294±68.15,0.241±0.36,0.289±0.45,0.275±0.39,0.952±0.17,-0.032±0.23
GradientAscent,29.48±35.36,0.278±0.37,0.131±0.46,0.231±0.32,1.267±0.04,0.002±0.05
SmoothGrad,66.883±58.16,0.331±0.4,0.321±0.49,0.364±0.36,0.639±0.08,-0.024±0.2
FusionGrad,58.514±57.8,0.344±0.39,0.336±0.48,0.357±0.35,0.855±0.09,-0.014±0.17
GradientShap,69.072±62.93,0.233±0.38,0.409±0.44,0.344±0.41,1.4±0.41,-0.023±0.24
IntegratedGradients,66.83±61.21,0.244±0.38,0.421±0.44,0.35±0.41,0.788±0.21,-0.029±0.23


In [10]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

def _as_float_array(value):
    if isinstance(value, (list, tuple, np.ndarray, pd.Series)):
        arr = np.asarray(value, dtype=float).reshape(-1)
    else:
        arr = np.asarray([value], dtype=float)
    return arr[np.isfinite(arr)]

def _format_val(x, precision=3):
    if np.isnan(x):
        return ""
    if abs(x) >= 1e4 or (abs(x) > 0 and abs(x) < 1e-3):
        return f"{x:.2e}"
    s = f"{x:.{precision}f}"
    return s.rstrip("0").rstrip(".") if "." in s else s

def _format_mean_std(arr, precision=3):
    if arr.size == 0:
        return ""
    mean = arr.mean()
    std = arr.std()
    mean_str = _format_val(mean, precision=precision)
    std_str = _format_val(std, precision=max(1, precision - 1))
    return f"{mean_str}±{std_str}"

def summarize_ablations_pickle(pkl_path, show=True):
    pkl_path = Path(pkl_path)
    if not pkl_path.exists():
        raise FileNotFoundError(f"File not found: {pkl_path}")

    df = pd.read_pickle(pkl_path)
    meta_cols = ["model_name", "parameter", "value", "default_value", "explainer"]
    metric_cols = [
        col
        for col in df.columns
        if col not in meta_cols and not col.startswith("is_default")
    ]
    group_cols = ["model_name", "parameter", "value", "default_value", "explainer"]

    summary_rows = []
    for _, row in df.iterrows():
        base = {col: row[col] for col in group_cols}
        for metric in metric_cols:
            arr = _as_float_array(row[metric])
            base[metric] = _format_mean_std(arr, precision=3)
        summary_rows.append(base)

    summary = (
        pd.DataFrame(summary_rows)
        .sort_values(["model_name", "parameter", "value", "explainer"])
        .reset_index(drop=True)
    )

    csv_path = pkl_path.with_name(pkl_path.stem + "_summary.csv")
    summary.to_csv(csv_path, index=False)

    print(f"Loaded: {pkl_path}")
    print(f"Saved CSV: {csv_path}")
    if show:
        display(summary)
    return summary, csv_path

def summarize_focus_pickle(focus_pkl_path, show=True):
    focus_pkl_path = Path(focus_pkl_path)
    if not focus_pkl_path.exists():
        results_dir = focus_pkl_path.parent
        available = [p.name for p in sorted(results_dir.glob("focus_model_name=*.pkl"))]
        raise FileNotFoundError(
            f"File not found: {focus_pkl_path}. Available Focus files: {available}"
        )

    focus_df = pd.read_pickle(focus_pkl_path)
    required_cols = {"explainer", "score"}
    missing_cols = required_cols.difference(focus_df.columns)
    if missing_cols:
        raise ValueError(
            f"Missing required columns in Focus results: {sorted(missing_cols)}"
        )

    focus_summary = (
        focus_df.groupby("explainer", dropna=False)["score"]
        .agg(["mean", "std", "count"])
        .sort_values("mean", ascending=False)
        .reset_index()
    )

    focus_csv_path = focus_pkl_path.with_name(focus_pkl_path.stem + "_summary.csv")
    focus_summary.to_csv(focus_csv_path, index=False)

    print(f"Loaded: {focus_pkl_path}")
    print(f"Saved CSV: {focus_csv_path}")
    if show:
        display(focus_summary)
    return focus_summary, focus_csv_path

def paired_differences_vs_default(ablations_pkl_path, show=True):
    ablations_pkl_path = Path(ablations_pkl_path)
    if not ablations_pkl_path.exists():
        raise FileNotFoundError(f"File not found: {ablations_pkl_path}")

    df = pd.read_pickle(ablations_pkl_path)
    meta_cols = {"model_name", "parameter", "value", "default_value", "explainer", "is_default"}
    metric_cols = [col for col in df.columns if col not in meta_cols]

    rows = []
    for explainer, df_expl in df.groupby("explainer", sort=False):
        if "is_default" in df_expl.columns and df_expl["is_default"].any():
            default_rows = df_expl[df_expl["is_default"] == True]
        else:
            default_rows = df_expl[
                (df_expl["parameter"] == "default") & (df_expl["value"] == "default")
            ]

        if len(default_rows) != 1:
            raise ValueError(
                f"Expected exactly one default row for explainer={explainer}, got {len(default_rows)}"
            )

        default_row = default_rows.iloc[0]
        for _, row in df_expl.iterrows():
            is_default_row = bool(row.get("is_default", False)) or (
                row["parameter"] == "default" and row["value"] == "default"
            )
            if is_default_row:
                continue

            setting = f"{row['parameter']}={row['value']}"
            for metric in metric_cols:
                x = _as_float_array(row[metric])
                y = _as_float_array(default_row[metric])
                n = min(len(x), len(y))
                if n == 0:
                    continue

                diff = x[:n] - y[:n]
                rows.append(
                    {
                        "model_name": row["model_name"],
                        "explainer": explainer,
                        "setting": setting,
                        "parameter": row["parameter"],
                        "value": row["value"],
                        "metric": metric,
                        "delta_mean": float(np.mean(diff)),
                        "delta_std": float(np.std(diff, ddof=1)) if n > 1 else np.nan,
                        "paired_count": int(n),
                    }
                )

    paired_df = pd.DataFrame(rows).sort_values(
        ["explainer", "parameter", "value", "metric"]
    ).reset_index(drop=True)

    paired_csv_path = ablations_pkl_path.with_name(
        ablations_pkl_path.stem + "_paired_diff_vs_default.csv"
    )
    paired_df.to_csv(paired_csv_path, index=False)

    paired_df["delta_mean_std"] = paired_df.apply(
        lambda r: f"{r['delta_mean']:.4f}±{r['delta_std']:.4f}" if np.isfinite(r['delta_std']) else f"{r['delta_mean']:.4f}",
        axis=1,
    )
    paired_pivot = (
        paired_df.assign(value_num=pd.to_numeric(paired_df["value"], errors="coerce"))
        .sort_values(["explainer", "parameter", "value_num", "value"])
        .pivot_table(
            index=["explainer", "setting"],
            columns="metric",
            values="delta_mean_std",
            aggfunc="first",
        )
        .reset_index()
    )

    print(f"Loaded: {ablations_pkl_path}")
    print(f"Saved paired difference CSV: {paired_csv_path}")
    print("Delta format: setting - default (mean±std across paired differences)")
    if show:
        display(paired_pivot)
    return paired_df, paired_pivot, paired_csv_path

In [11]:
summary_resnet_ablations, csv_resnet_ablations = summarize_ablations_pickle(
    "results/quantus_ablations_model_name=resnet50_n_batches=25_test_mode=test.pkl"
)

Loaded: results\quantus_ablations_model_name=resnet50_n_batches=25_test_mode=test.pkl
Saved CSV: results\quantus_ablations_model_name=resnet50_n_batches=25_test_mode=test_summary.csv


,model_name,parameter,value,default_value,explainer,infidelity,faithfulness_correlation,faithfulness_estimate,random_logit
0,resnet50,K,1,5,PullbackAscent,6.024±5.72,0.398±0.39,0.466±0.39,-0.019±0.43
1,resnet50,K,2,5,PullbackAscent,5.534±5.1,0.405±0.39,0.433±0.41,0.152±0.31
2,resnet50,K,3,5,PullbackAscent,5.37±4.91,0.398±0.39,0.419±0.42,0.199±0.26
3,resnet50,K,10,5,PullbackAscent,5.652±5.79,0.354±0.38,0.358±0.46,0.214±0.14
4,resnet50,alpha,5,20,PullbackAscent,5.459±5.02,0.409±0.38,0.415±0.41,0.266±0.24
5,resnet50,alpha,10,20,PullbackAscent,5.357±4.89,0.397±0.38,0.404±0.42,0.263±0.21
6,resnet50,alpha,40,20,PullbackAscent,5.68±7.27,0.362±0.38,0.377±0.46,0.17±0.18
7,resnet50,default,default,default,PullbackAscent,5.338±4.86,0.381±0.38,0.389±0.45,0.228±0.2
8,resnet50,default,default,default,SoftPullback,6.024±5.75,0.398±0.39,0.466±0.39,-0.019±0.43
9,resnet50,tau_maxpool,0.01,0.3,PullbackAscent,5.339±4.86,0.364±0.39,0.328±0.48,0.196±0.18


In [ ]:
summary_resnet_default, csv_resnet_default = summarize_ablations_pickle(
    "results/quantus_ablations_default_model_name=resnet50_n_batches=25_test_mode=test.pkl"
)

Loaded: results\quantus_ablations_model_name=pvt_v2_b1_n_batches=25_test_mode=test.pkl
Saved CSV: results\quantus_ablations_model_name=pvt_v2_b1_n_batches=25_test_mode=test_summary.csv


,model_name,parameter,value,default_value,explainer,infidelity,faithfulness_correlation,faithfulness_estimate,random_logit
0,pvt_v2_b1,K,1,5,PullbackAscent,6.44±6.2,0.097±0.33,0.152±0.39,-0.018±0.38
1,pvt_v2_b1,K,2,5,PullbackAscent,2.8±3.01,0.124±0.36,0.198±0.41,0.04±0.21
2,pvt_v2_b1,K,3,5,PullbackAscent,1.883±1.26,0.134±0.37,0.209±0.42,0.07±0.16
3,pvt_v2_b1,K,10,5,PullbackAscent,1.571±1.14,0.12±0.36,0.184±0.42,0.139±0.07
4,pvt_v2_b1,alpha,5,20,PullbackAscent,1.993±1.68,0.134±0.36,0.144±0.43,0.107±0.13
5,pvt_v2_b1,alpha,10,20,PullbackAscent,1.737±1.21,0.133±0.37,0.176±0.43,0.113±0.12
6,pvt_v2_b1,alpha,40,20,PullbackAscent,1.657±1.32,0.136±0.37,0.234±0.41,0.104±0.11
7,pvt_v2_b1,default,default,default,PullbackAscent,1.693±1.41,0.133±0.37,0.202±0.43,0.111±0.11
8,pvt_v2_b1,default,default,default,SoftPullback,6.348±5.98,0.097±0.33,0.152±0.39,-0.018±0.38
9,pvt_v2_b1,tau_attention,0.5,1.0,PullbackAscent,1.743±1.3,0.13±0.36,0.195±0.43,0.095±0.1


In [ ]:
focus_summary_resnet, focus_csv_resnet = summarize_focus_pickle(
    "results/focus_model_name=resnet50.pkl"
)

Loaded: results\focus_model_name=resnet50.pkl
Saved CSV: results\focus_model_name=resnet50_summary.csv


,explainer,mean,std,count
0,GuidedGradCam,0.849295,0.176164,500
1,SoftPullback,0.748278,0.134405,500
2,PullbackAscent,0.730507,0.129333,500
3,SmoothPullback,0.718342,0.149743,500
4,FusionPullback,0.696509,0.150471,500
5,SmoothGrad,0.640846,0.123955,500
6,FusionGrad,0.634871,0.119925,500
7,Gradient,0.633657,0.139535,500
8,IntegratedGradients,0.622721,0.167035,500
9,DeepLift,0.621046,0.175279,500


In [ ]:
focus_summary_pvt, focus_csv_pvt = summarize_focus_pickle(
    "results/focus_model_name=pvt_v2_b1.pkl"
)

Loaded: results\focus_model_name=pvt_v2_b1.pkl
Saved CSV: results\focus_model_name=pvt_v2_b1_summary.csv


,explainer,mean,std,count
0,SmoothPullback,0.634890,0.102580,500
1,SoftPullback,0.624963,0.164957,500
2,FusionPullback,0.615664,0.104791,500
3,FusionGrad,0.605290,0.104144,500
4,SmoothGrad,0.592187,0.111431,500
5,Gradient,0.560886,0.170342,500
6,IntegratedGradients,0.559527,0.177091,500
7,GradientShap,0.553672,0.179238,500
8,PullbackAscent,0.553121,0.094839,500
9,DeepLift,0.550655,0.203432,500


In [12]:
paired_df_resnet, paired_pivot_resnet, paired_csv_resnet = paired_differences_vs_default(
    "results/quantus_ablations_model_name=resnet50_n_batches=25_test_mode=test.pkl"
)
# paired_df_pvt, paired_pivot_pvt, paired_csv_pvt = paired_differences_vs_default(
#     "results/quantus_ablations_model_name=pvt_v2_b1_n_batches=25_test_mode=test.pkl"
# )

Loaded: results\quantus_ablations_model_name=resnet50_n_batches=25_test_mode=test.pkl
Saved paired difference CSV: results\quantus_ablations_model_name=resnet50_n_batches=25_test_mode=test_paired_diff_vs_default.csv
Delta format: setting - default (mean±std across paired differences)


metric,explainer,setting,faithfulness_correlation,faithfulness_estimate,infidelity,random_logit
0,PullbackAscent,K=1,0.0165±0.1276,0.0761±0.2277,0.6854±3.0195,-0.2464±0.2604
1,PullbackAscent,K=10,-0.0270±0.0776,-0.0318±0.1619,0.3136±2.9970,-0.0133±0.0703
2,PullbackAscent,K=2,0.0234±0.0835,0.0431±0.1688,0.1958±1.4689,-0.0758±0.1319
3,PullbackAscent,K=3,0.0165±0.0520,0.0300±0.1338,0.0318±0.7931,-0.0284±0.0708
4,PullbackAscent,alpha=10,0.0158±0.0490,0.0150±0.1267,0.0189±0.7609,0.0354±0.0453
5,PullbackAscent,alpha=40,-0.0190±0.0587,-0.0125±0.1442,0.3421±5.2001,-0.0581±0.0415
6,PullbackAscent,alpha=5,0.0277±0.0832,0.0260±0.1671,0.1204±1.3798,0.0380±0.0888
7,PullbackAscent,tau_maxpool=0.01,-0.0177±0.0663,-0.0618±0.2088,0.0011±0.6550,-0.0313±0.0415
8,PullbackAscent,tau_maxpool=0.5,0.0043±0.0201,0.0194±0.0989,0.0580±0.2961,0.0027±0.0146
9,PullbackAscent,tau_relu=0.3,-0.0226±0.1688,-0.0484±0.3056,3.0005±7.9787,-0.1563±0.1414


In [ ]:
summary_pvt_ablations, csv_pvt_ablations = summarize_ablations_pickle(
    "results/quantus_ablations_model_name=pvt_v2_b1_n_batches=25_test_mode=test.pkl"
)